# exp109_typewell_neighbor_prior_features train

Fold-safe audit of native typewell overlap neighbor priors as weak corrections for PF/Beam/likPF candidates.

## Contents

1. Setup and configuration
2. Input artifact checks
3. Fold-safe neighbor prior audit
4. Metrics and artifacts

## 1. Setup and configuration

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, get_nested, load_config
from typewell_neighbor_prior_features import (
    EXP065_CLUSTER_ASSIGNMENTS,
    EXP099_FEATURE_CACHE,
    find_artifact,
    parse_group_methods,
    run_audit,
)

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
group_methods = parse_group_methods(config)

print("Experiment:", EXPERIMENT_NAME)
print("Route:", get_nested(config, "experiment.route"))
print("Parent:", get_nested(config, "lineage.parent"))
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Group methods:", [method.name for method in group_methods])

## 2. Input artifact checks

In [ ]:
feature_cache = find_artifact(
    EXP099_FEATURE_CACHE,
    get_nested(config, "data.exp099_train_feature_cache_local"),
)
cluster_assignments = find_artifact(
    EXP065_CLUSTER_ASSIGNMENTS,
    get_nested(config, "data.exp065_cluster_assignments_local"),
)

feature_header = pd.read_csv(feature_cache, nrows=0).columns.tolist()
cluster_preview = pd.read_csv(cluster_assignments, nrows=5)

print("Feature cache:", feature_cache)
print("Feature columns:", feature_header[:16], "...", len(feature_header))
print("Cluster assignments:", cluster_assignments)
display(cluster_preview)

## 3. Fold-safe neighbor prior audit

In [ ]:
summary = run_audit(config=config, paths=paths)
print(json.dumps(summary["best_candidate"], indent=2, sort_keys=True))
print("Delta best - likPF RMSE:", summary["delta_best_minus_likpf_rmse"])

## 4. Metrics and artifacts

In [ ]:
metrics_path = Path(summary["artifacts"]["candidate_metrics"])
bucket_path = Path(summary["artifacts"]["bucket_metrics"])
by_well_path = Path(summary["artifacts"]["by_well"])

metrics = pd.read_csv(metrics_path)
bucket_metrics = pd.read_csv(bucket_path)
by_well = pd.read_csv(by_well_path)

display(metrics.head(20))
display(bucket_metrics[bucket_metrics["candidate"].isin(metrics.head(5)["candidate"])].head(40))
display(by_well.sort_values("rmse", ascending=False).head(20))

print("Summary:", summary["artifacts"]["summary"])
print("OOF predictions:", summary["artifacts"]["oof_predictions"])
print("Feature schema:", summary["artifacts"]["feature_schema"])